# U8: what out-of-bounds stores do on a GPU, and whether an output check notices

Runs every out-of-bounds-store mutant (oracle label `oob`, witness access `store`: 11 mutants over 6 kernels), the 16
out-of-bounds-load mutants and the 8 reference kernels through `triton_equiv.oobwrite` at each kernel's odd checker size,
10 input seeds, in two layouts: `adjacent` (sentinels directly before and after the output in one allocation: every
element written outside) and `pool` (output and sentinel as separate `torch.empty` calls, as a benchmark allocates them).
The checker verdict is `torch.allclose` against the PyTorch reference at KernelBench's 1e-2 and at torch's defaults, the
comparison KernelGYM's `kernel_simple` makes (`docs/kernelgym_audit.md`). One process per mutant, since an illegal-address
fault poisons the CUDA context. Output: `/kaggle/working/u8_oob_writes.json`.

Kaggle: GPU T4, internet off is fine; private dataset `rajulkabir/tileproof-src-u8` attached (SOURCE.txt names the commit).


In [ ]:
import glob, json, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
HARD_STOP_S = 3600   # the whole run is expected to take minutes; stop starting mutants after an hour
os.environ["TRITON_INTERPRET"] = "0"
SRC = "/kaggle/working/tileproof_src"
tars = glob.glob("/kaggle/input/**/tileproof_src.tar.gz", recursive=True)
srcs = glob.glob("/kaggle/input/**/SOURCE.txt", recursive=True)
os.makedirs(SRC, exist_ok=True)
if tars:
    with tarfile.open(tars[0]) as t:
        t.extractall(SRC)
elif srcs:  # Kaggle unpacked the tarball
    shutil.copytree(os.path.dirname(srcs[0]), SRC, dirs_exist_ok=True)
else:
    raise SystemExit("dataset missing: " + str(glob.glob("/kaggle/input/**", recursive=True)[:50]))
print(open(SRC + "/SOURCE.txt").read())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
sys.path[:0] = [SRC]
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(0))


In [ ]:
from triton_equiv import oobwrite
from triton_equiv.kernels import KERNELS
labels = json.load(open(SRC + "/labels.json"))["labels"]
STORE, LOAD = oobwrite.store_oob_ids(labels), oobwrite.load_oob_ids(labels)
REFS = [f"{k}:ref" for k in KERNELS]
OUT = "/kaggle/working/u8"; os.makedirs(OUT, exist_ok=True)
rows, crashes = [], {}
for mid in REFS + STORE + LOAD:
    if time.time() - T0 > HARD_STOP_S:
        crashes[mid] = "not run: hard stop"; continue
    path = f"{OUT}/{mid.replace(':', '_')}.json"
    r = subprocess.run([sys.executable, "-m", "triton_equiv.oobwrite", mid, "--trials", "10", "--device", "cuda",
                        "--out", path], capture_output=True, text=True, timeout=900)
    if r.returncode != 0:
        crashes[mid] = f"exit {r.returncode}: {r.stderr[-600:]}"
        print(mid, "CRASH", r.returncode, r.stderr[-200:]); continue
    rs = json.load(open(path)); rows += rs
    print(mid, json.dumps(oobwrite.summary(rs)[mid]))


In [ ]:
summ = oobwrite.summary(rows)
def group(ids):
    return {"n": len(ids), "accepted_kernelbench_all_trials": sum(summ.get(i, {}).get("accept", {}).get("kernelbench") == 1 for i in ids),
            "wrote_outside": sum((summ.get(i, {}).get("adjacent_written") or 0) > 0 for i in ids),
            "accepted_and_wrote_outside": sum(summ.get(i, {}).get("accept", {}).get("kernelbench") == 1
                                              and (summ.get(i, {}).get("adjacent_written") or 0) > 0 for i in ids),
            "pool_sentinel_hit": sum((summ.get(i, {}).get("pool_hit") or 0) > 0 for i in ids)}
result = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "triton": triton.__version__,
          "source": open(SRC + "/SOURCE.txt").read(), "seconds": round(time.time() - T0),
          "groups": {"reference": group(REFS), "store_oob": group(STORE), "load_oob": group(LOAD)},
          "summary": summ, "crashes": crashes, "rows": rows}
json.dump(result, open("/kaggle/working/u8_oob_writes.json", "w"), indent=1)
print(json.dumps(result["groups"], indent=1)); print(len(crashes), "crashes")
